In [1]:
!pip install psutil


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import psutil
import os

vm = psutil.virtual_memory()
sw = psutil.swap_memory()

print("=== System Memory ===")

print(f"Total RAM : {vm.total / 1024**3:.2f} GB")
print(f"Used RAM : {vm.used / 1024**3:.2f} GB ({vm.percent}%)")
print(f"Available : {vm.available / 1024**3:.2f} GB")

print(f"Swap Total : {sw.total / 1024**3:.2f} GB")
print(f"Swap Used : {sw.used / 1024**3:.2f} GB")

print("\n=== Current Process Memory ===")

p = psutil.Process(os.getpid())
mi = p.memory_info()

print(f"RSS (physical) : {mi.rss / 1024**2:.2f} MB")
print(f"VMS (virtual) : {mi.vms / 1024**2:.2f} MB")

=== System Memory ===
Total RAM : 3.70 GB
Used RAM : 3.21 GB (86.6%)
Available : 0.50 GB
Swap Total : 7.52 GB
Swap Used : 4.85 GB

=== Current Process Memory ===
RSS (physical) : 68.80 MB
VMS (virtual) : 72.36 MB


In [3]:
PAGE_SIZE = 4096

page_table = {
    0: 3,
    1: 7,
    2: None,
    3: 1
}

def translate(virtual_address):

    page = virtual_address // PAGE_SIZE
    offset = virtual_address % PAGE_SIZE

    if page not in page_table:
        raise ValueError(f"Segfault: page {page} not mapped")

    frame = page_table[page]

    if frame is None:
        print(f"Page fault! OS loads page {page} from disk")
        page_table[page] = 5
        frame = 5

    physical = frame * PAGE_SIZE + offset

    print(f"VA {virtual_address:#010x} -> page {page}, offset {offset} -> frame {frame} -> PA {physical:#010x}")

    return physical


for va in [0, 4096, 8000, 12288]:
    translate(va)

VA 0x00000000 -> page 0, offset 0 -> frame 3 -> PA 0x00003000
VA 0x00001000 -> page 1, offset 0 -> frame 7 -> PA 0x00007000
VA 0x00001f40 -> page 1, offset 3904 -> frame 7 -> PA 0x00007f40
VA 0x00003000 -> page 3, offset 0 -> frame 1 -> PA 0x00001000


In [4]:
for va in [0, 4096, 8000, 12288, 16384, 20000]:
    translate(va)

VA 0x00000000 -> page 0, offset 0 -> frame 3 -> PA 0x00003000
VA 0x00001000 -> page 1, offset 0 -> frame 7 -> PA 0x00007000
VA 0x00001f40 -> page 1, offset 3904 -> frame 7 -> PA 0x00007f40
VA 0x00003000 -> page 3, offset 0 -> frame 1 -> PA 0x00001000


ValueError: Segfault: page 4 not mapped

In [5]:
page_table = {
    0: 3,
    1: 7,
    2: None,
    3: None
}

translate(12288)

Page fault! OS loads page 3 from disk
VA 0x00003000 -> page 3, offset 0 -> frame 5 -> PA 0x00005000


20480

In [6]:
import psutil
import os

process = psutil.Process(os.getpid())

before = process.memory_info().rss / 1024**2
print("Memory before allocation:", before, "MB")

big_list = [0] * 10_000_000

after = process.memory_info().rss / 1024**2
print("Memory after allocation:", after, "MB")

print("Memory consumed:", after - before, "MB")

Memory before allocation: 85.0859375 MB
Memory after allocation: 161.3828125 MB
Memory consumed: 76.296875 MB


In [7]:
pages = [1,2,3,4,1,2,5,1,2,3]

frames = []
capacity = 4
page_faults = 0

for page in pages:

    if page not in frames:
        page_faults += 1

        if len(frames) < capacity:
            frames.append(page)
        else:
            frames.pop(0)
            frames.append(page)

    else:
        frames.remove(page)
        frames.append(page)

    print("Frames:", frames)

print("\nTotal Page Faults:", page_faults)

Frames: [1]
Frames: [1, 2]
Frames: [1, 2, 3]
Frames: [1, 2, 3, 4]
Frames: [2, 3, 4, 1]
Frames: [3, 4, 1, 2]
Frames: [4, 1, 2, 5]
Frames: [4, 2, 5, 1]
Frames: [4, 5, 1, 2]
Frames: [5, 1, 2, 3]

Total Page Faults: 6
